# Cohort — clean SoH plots + characterisation parameter distributions

**Locked training cohort (13 cells)** — from `training_cohort.yaml`:

- **CALB** (5): 0003, 0008, 0009, 0010, 0015 · DoD ∈ {0_100, 15_100}, SoH_first ≥ 0.60
- **EVE** (3): 0002, 0004, 0008 · 150-cy trajectories, SoH_first ≥ 0.60 · (0003 dropped, no GITT)
- **REPT** (5): 0004, 0007, 0012, 0046, 0057 · noise-ranked, smoothable

**Final characterisation parameter set** (per test):

| Test | Parameter | Symbol | Role in Phase 1 |
|---|---|---|---|
| RPT | discharge capacity (÷ nameplate) | `Q/Q_nom` | initial-SoH prior + capacity check |
| HPPC | ohmic resistance median | `R0` | contact + electrolyte R prior |
| HPPC | charge-transfer resistance median | `R1` | interfacial resistance |
| GITT | diffusion time constant median | `τ_diff` | solid-phase Li diffusion prior |
| GITT | dV/d√t median | `dV/√t` | diffusion-slope diagnostic |
| OCV | plateau voltage at SoC=0.5 | `V_plat` | LFP-topology invariance check (control) |
| Longterm | starting state-of-health | `SoH_first` | initial state for Phase 2 fit |
| Longterm | fade over the record | `fade_pp` | Phase 2 signal strength |

**Optional (diagnostic only)**: self-discharge dSoC/day — 8 of 13 cells have it.

**Excluded**: DCIR (variable SoC-sampling protocol across batches — HPPC R0 covers the resistance signal).

In [1]:
import sys, yaml, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Markdown, display

warnings.filterwarnings('ignore')
HERE = Path('/home/hj/Desktop/PINNs/Voltaris/Data_Exploration')
sys.path.insert(0, str(HERE))
from extract import calb_longterm_soh

# Load cohort manifest
manifest = yaml.safe_load((HERE / 'training_cohort.yaml').read_text())
FINAL_COHORT = {k: [str(c).zfill(4) for c in v] for k, v in manifest['cohort'].items()}
print(f'Cohort loaded: {manifest["n_cells_total"]} cells across {len(FINAL_COHORT)} makes')
for m, cells in FINAL_COHORT.items():
    print(f'  {m}: {cells}')

MAKE_COLOR = {'CALB': '#c94a3c', 'EVE': '#4dab5c', 'REPT': '#3c7cc9'}
NAMEPLATE  = {'CALB': 72.0, 'EVE': 105.0, 'REPT': 150.0}
SMOOTH_WIN = manifest['smoothing']['window']

Cohort loaded: 13 cells across 3 makes
  CALB: ['0003', '0008', '0009', '0010', '0015']
  EVE: ['0002', '0004', '0008']
  REPT: ['0004', '0007', '0012', '0046', '0057']


In [2]:
def _load_parquet(path):
    df = pd.read_parquet(path)
    df['cell'] = df['cell_id'].astype(str).str.zfill(4)
    return {c: sub.sort_values('global_cycle').reset_index(drop=True).rename(columns={'global_cycle': 'cycle'})[['cycle','soh']]
            for c, sub in df.groupby('cell')}

traces = {
    'CALB': calb_longterm_soh(),
    'EVE':  _load_parquet('/home/hj/Desktop/PINNs/soh/data/canonical/eve.parquet'),
    'REPT': _load_parquet('/home/hj/Desktop/PINNs/soh/data/canonical/rept.parquet'),
}

def _smoothed(soh, window=SMOOTH_WIN):
    return pd.Series(soh).rolling(window, center=True, min_periods=3).median().values

def _stats(soh):
    s = pd.Series(soh).dropna().values
    return dict(soh_first=float(np.mean(s[:5])),
                soh_last=float(np.mean(s[-5:])),
                fade_pp=float((np.mean(s[:5]) - np.mean(s[-5:])) * 100),
                n_cy=len(s))

## 1. Clean SoH trajectories — 13 cohort cells

Smoothed with rolling median (window = 7) to suppress single-cycle spikes. Coloured by make; hover shows cell + cycle + SoH.

In [3]:
fig = go.Figure()
for make in ['CALB', 'EVE', 'REPT']:
    color = MAKE_COLOR[make]
    first = True
    for cid in FINAL_COHORT[make]:
        df = traces[make][cid]
        y = _smoothed(df['soh'].values)
        fig.add_trace(go.Scatter(
            x=df['cycle'], y=y, mode='lines',
            name=make if first else None,
            legendgroup=make, showlegend=first,
            line=dict(color=color, width=1.8), opacity=0.85,
            hovertemplate=(f'<b>{make} · {cid}</b><br>cycle %{{x}}<br>'
                            'SoH %{y:.3f}<extra></extra>'),
        ))
        first = False

for y_ref, label in [(1.00, 'nominal'), (0.80, 'EoL 0.80'), (0.60, 'floor 0.60')]:
    fig.add_hline(y=y_ref, line=dict(color='grey', width=0.7,
                                        dash='dot' if y_ref != 0.60 else 'dash'),
                  annotation_text=label, annotation_position='right',
                  annotation_font_size=9)

fig.update_layout(
    title=dict(text='13-cell training cohort — SoH vs cycle (rolling-median smoothed, w=7)',
                x=0.02, font_size=14),
    xaxis=dict(title='cycle', gridcolor='rgba(0,0,0,0.08)'),
    yaxis=dict(title='SoH', range=[0.55, 1.05], gridcolor='rgba(0,0,0,0.08)'),
    legend=dict(orientation='v', x=1.02, y=1.0, title='make', font_size=11),
    plot_bgcolor='white', width=1050, height=520,
    margin=dict(l=60, r=140, t=60, b=50),
)
fig.show()

## 2. Cohort characterisation parameters — table

One row per cell. Values from `extract.py` (RAW, computed from raw `detail` CSVs).

In [4]:
# Load RAW scalars
sc = pd.read_csv(HERE / 'characterization_scalars.csv')
sc['cell_id'] = sc['cell_id'].astype(str).str.zfill(4)
sc = sc[sc.apply(lambda r: r['cell_id'] in FINAL_COHORT.get(r['make'], []), axis=1)].copy()
sc['nameplate']       = sc['make'].map(NAMEPLATE)
sc['q_over_nom']      = sc['capacity_Ah'] / sc['nameplate']

# Attach SoH_first + fade from Longterm traces
def _soh_first_fade(row):
    df = traces[row['make']].get(row['cell_id'])
    if df is None or df.empty:
        return pd.Series({'soh_first': np.nan, 'fade_pp': np.nan, 'n_cy': np.nan})
    s = df['soh'].dropna().values
    return pd.Series({'soh_first': float(np.mean(s[:5])),
                       'fade_pp':   float((np.mean(s[:5]) - np.mean(s[-5:])) * 100),
                       'n_cy':      int(len(s))})
sc[['soh_first','fade_pp','n_cy']] = sc.apply(_soh_first_fade, axis=1)

# Final column set (rename for clarity)
final = sc[['make','cell_id','q_over_nom','HPPC_R0_mOhm','HPPC_R1_mOhm',
             'GITT_tau_diff_s','dV_per_sqrt_t','V_plateau',
             'soh_first','fade_pp','n_cy',
             'self_disch_dsoc_per_day']].copy()
final.columns = ['make','cell','Q/Q_nom','R0 (mΩ)','R1 (mΩ)',
                  'τ_diff (s)','dV/√t (V/√s)','V_plat (V)',
                  'SoH_first','fade (pp)','n_cy',
                  'dSoC/day (optional)']
final = final.sort_values(['make','cell']).reset_index(drop=True)

pd.set_option('display.max_columns', None)
display(Markdown('### Final cohort characterisation table'))
display(final.round({'Q/Q_nom':3,'R0 (mΩ)':2,'R1 (mΩ)':2,'τ_diff (s)':0,
                       'dV/√t (V/√s)':5,'V_plat (V)':3,'SoH_first':3,'fade (pp)':2,
                       'dSoC/day (optional)':5}))
final.to_csv(HERE / 'cohort_parameters_final.csv', index=False)
print(f'Wrote: {HERE / "cohort_parameters_final.csv"}')

### Final cohort characterisation table

,make,cell,Q/Q_nom,R0 (mΩ),R1 (mΩ),τ_diff (s),dV/√t (V/√s),V_plat (V),SoH_first,fade (pp),n_cy,dSoC/day (optional)
0,CALB,0003,0.791,1.05,0.58,878.0,-0.00082,3.267,0.636,4.28,402.0,0.00784
1,CALB,0008,0.707,1.37,0.77,876.0,-0.00096,3.259,0.600,6.11,402.0,NaN
2,CALB,0009,0.747,1.12,0.64,872.0,-0.00089,3.265,0.702,5.20,402.0,NaN
3,CALB,0010,0.730,1.33,0.72,867.0,-0.00084,3.264,0.766,4.42,401.0,NaN
4,CALB,0015,0.778,1.05,0.60,879.0,-0.00091,3.268,0.795,2.97,402.0,0.00907
5,EVE,0002,0.993,1.72,0.35,814.0,-0.00108,3.266,0.989,0.19,150.0,NaN
6,EVE,0004,0.989,0.79,0.35,844.0,-0.00092,3.268,0.961,0.13,150.0,NaN
7,EVE,0008,0.991,1.78,0.38,819.0,-0.00103,3.265,0.955,1.51,150.0,0.00641
8,REPT,0004,1.024,0.81,0.24,834.0,-0.00126,3.268,0.632,0.40,205.0,0.00495
9,REPT,0007,0.988,1.05,0.25,822.0,-0.00129,3.264,0.990,2.56,205.0,0.00773


Wrote: /home/hj/Desktop/PINNs/Voltaris/Data_Exploration/cohort_parameters_final.csv


## 3. Parameter distributions across the cohort

One panel per parameter. Points are individual cells, coloured by make. Dashed line = physical bound (where applicable). Look for:

- **Adequate spread within each make** — DE fits need diversity in the θ space
- **Cross-make separation** — if a parameter is nearly identical across all makes, it's not carrying supplier-specific information
- **Outliers** — any cell far from its make's median may have an extraction problem

In [5]:
PARAMS = [
    ('Q/Q_nom',    'Q_rpt / nameplate',      (0.40, 1.05)),
    ('R0 (mΩ)',    'HPPC R0 median (mΩ)',    (0.15, 5.0)),
    ('R1 (mΩ)',    'HPPC R1 median (mΩ)',    None),
    ('τ_diff (s)', 'GITT τ_diff median (s)', (10, 10000)),
    ('dV/√t (V/√s)', 'GITT dV/d√t (V/√s)',    (-1e-2, -1e-4)),
    ('V_plat (V)', 'OCV V_plateau (V)',       (3.15, 3.35)),
    ('SoH_first',  'Longterm SoH_first',      (0.60, 1.00)),
    ('fade (pp)',  'Longterm fade (pp)',      None),
    ('dSoC/day (optional)', 'Self-disch dSoC/day (optional)', None),
]

ncols = 3
nrows = int(np.ceil(len(PARAMS) / ncols))
fig = make_subplots(rows=nrows, cols=ncols,
                     subplot_titles=[p[1] for p in PARAMS],
                     vertical_spacing=0.10, horizontal_spacing=0.10)

makes = ['CALB', 'EVE', 'REPT']
y_pos = {m: i for i, m in enumerate(makes)}

for i, (col, label, bounds) in enumerate(PARAMS):
    row = i // ncols + 1; c = i % ncols + 1
    for m in makes:
        sub = final[final['make'] == m]
        vals = sub[col].values
        cells = sub['cell'].values
        y_jitter = np.full(len(vals), y_pos[m], dtype=float) + \
                    np.random.default_rng(abs(hash(m+str(len(vals)))) % (2**32)).uniform(-0.15, 0.15, len(vals))
        fig.add_trace(go.Scatter(
            x=vals, y=y_jitter, mode='markers+text',
            marker=dict(size=10, color=MAKE_COLOR[m],
                         line=dict(color='white', width=1.2)),
            text=cells, textposition='top center', textfont_size=8,
            showlegend=False,
            hovertemplate=(f'<b>{m} · %{{text}}</b><br>'
                            f'{label}: %{{x}}<extra></extra>'),
        ), row=row, col=c)
    if bounds is not None:
        lo, hi = bounds
        if np.isfinite(lo):
            fig.add_vline(x=lo, row=row, col=c,
                           line=dict(color='grey', width=0.9, dash='dash'))
        if np.isfinite(hi):
            fig.add_vline(x=hi, row=row, col=c,
                           line=dict(color='grey', width=0.9, dash='dash'))
    fig.update_yaxes(tickmode='array',
                      tickvals=list(y_pos.values()),
                      ticktext=list(y_pos.keys()),
                      range=[-0.6, 2.6],
                      gridcolor='rgba(0,0,0,0.05)', row=row, col=c)
    fig.update_xaxes(gridcolor='rgba(0,0,0,0.08)', row=row, col=c)

fig.update_layout(
    title=dict(text='Characterisation parameter distributions across the 13-cell cohort',
                x=0.02, font_size=13),
    height=300 * nrows + 80, width=1150,
    plot_bgcolor='white',
    margin=dict(l=60, r=30, t=80, b=40),
)
fig.show()

## 4. Per-make parameter summary — median · range · spread

How much within-make variance do we have? Wide range = good for DE identifiability; near-zero range = the make will produce a degenerate θ cluster.

In [6]:
num_cols = [c for c, _, _ in PARAMS]
summary = final.groupby('make')[num_cols].agg(['median','min','max']).round(4)
display(Markdown('### Per-make aggregates (median · min · max)'))
display(summary)

# Also: normalised spread (max-min) per (make, param) — quick eyeball of within-make diversity
spread = final.groupby('make')[num_cols].agg(lambda x: x.max()-x.min()).round(4)
display(Markdown('### Within-make spread (max − min)'))
display(spread)

### Per-make aggregates (median · min · max)

Q/Q_nom                 R0 (mΩ)                 R1 (mΩ)                  \
      median     min     max  median     min     max  median     min     max   
make                                                                           
CALB  0.7469  0.7069  0.7913  1.1223  1.0459  1.3702  0.6379  0.5821  0.7691   
EVE   0.9906  0.9887  0.9930  1.7172  0.7944  1.7779  0.3527  0.3469  0.3769   
REPT  0.9919  0.9714  1.0244  0.8103  0.7398  1.0530  0.2299  0.2210  0.2478   

     τ_diff (s)                     dV/√t (V/√s)                 V_plat (V)  \
         median       min       max       median     min     max     median   
make                                                                          
CALB   876.2968  866.8087  879.3841      -0.0009 -0.0010 -0.0008     3.2646   
EVE    819.4790  813.9013  843.9143      -0.0010 -0.0011 -0.0009     3.2659   
REPT   831.2361  822.0450  840.2243      -0.0013 -0.0014 -0.0013     3.2672   

                     SoH_first                 fade (pp)                  \
         min     max    median     min     max    median     min     max   
make                                                                       
CALB  3.2592  3.2682    0.7019  0.6004  0.7951    4.4239  2.9744  6.1083   
EVE   3.2654  3.2678    0.9608  0.9547  0.9890    0.1918  0.1305  1.5090   
REPT  3.2642  3.2682    0.6323  0.6195  0.9904    0.7619  0.2697  2.5580   

     dSoC/day (optional)                  
                  median     min     max  
make                                      
CALB              0.0085  0.0078  0.0091  
EVE               0.0064  0.0064  0.0064  
REPT              0.0051  0.0043  0.0077

### Within-make spread (max − min)

,Q/Q_nom,R0 (mΩ),R1 (mΩ),τ_diff (s),dV/√t (V/√s),V_plat (V),SoH_first,fade (pp),dSoC/day (optional)
make,,,,,,,,,
CALB,0.0844,0.3243,0.1870,12.5754,0.0001,0.0090,0.1947,3.1339,0.0012
EVE,0.0044,0.9835,0.0300,30.0130,0.0002,0.0025,0.0343,1.3785,0.0000
REPT,0.0530,0.3132,0.0268,18.1793,0.0001,0.0040,0.3709,2.2883,0.0035


## 5. Parameter correlation matrix

Which parameters are collinear? Highly-correlated parameters (|r| > 0.9) don't add independent information — worth knowing when we design the fingerprint.

In [7]:
corr_cols = ['Q/Q_nom','R0 (mΩ)','R1 (mΩ)','τ_diff (s)','dV/√t (V/√s)',
              'V_plat (V)','SoH_first','fade (pp)']
corr = final[corr_cols].corr(method='pearson', min_periods=3).round(2)

fig = go.Figure(data=go.Heatmap(
    z=corr.values,
    x=corr.columns, y=corr.index,
    text=corr.values, texttemplate='%{text:+.2f}',
    colorscale='RdBu', zmid=0, zmin=-1, zmax=1,
    colorbar=dict(title='Pearson r'),
))
fig.update_layout(
    title=dict(text='Parameter correlation matrix (13-cell cohort, RAW)',
                x=0.02, font_size=13),
    width=780, height=650,
    plot_bgcolor='white',
    margin=dict(l=100, r=40, t=60, b=100),
)
fig.update_xaxes(tickangle=-30)
fig.show()

## 6. What to look for downstream

This cohort feeds Phase 1 (per-cell BOL identification). Each cell gets fitted separately with PyBaMM DFN + `Prada2013` initial parameters, then Phase 2 DE-fits SEI/plating/LAM against Longterm SoH.

**Green flags (things that suggest Phase 1 will work)**:

- Every cohort cell has OCV + GITT + HPPC + RPT + Longterm on disk (verified by `training_cohort.yaml`)
- SoH_first spans 0.60–0.99 across the cohort — three aging regimes represented
- V_plateau consistent within each make (LFP topology invariant → char is repeatable)

**Yellow flags (things to keep an eye on)**:

- EVE cluster at SoH_first ≈ 0.96–0.99 — the 3 EVE cells will produce near-identical BOL θ, low identifiability of aging params
- REPT bimodal SoH_first (0.62 / 0.99) — no middle-aged REPT for the mid-SoH regime
- SoH_first ∈ [0.80, 0.95] band empty across the whole cohort — will need to fill via corpus perturbation on the initial-SoH axis

**Optional-parameter caveat**:

- Self-discharge dSoC/day is available for only 8 of 13 cells (2 CALB + 1 EVE + 5 REPT). Used as a data-quality diagnostic in the correlation notebook, not as a Phase 1 gate. If we later need self-disch as a fingerprint feature, we'll need to impute for the missing 5 cells.